**`03_ingest_global_administrative_units`**

Import geometries from the Global Administrative Database (GADM).

Prepare reference tables for global administrative units
(countries, states, counties, etc.).

This notebook also documents the creation of the recipe `admin.openplaces.2026` derived from ISO and GADM



# Administrative units identifiers
``openplaces`` organizes its data by ``admin_id`` (data class: ``AdminId``).

``admin_id`` is a geographical administrative index with hierarchical ``.levels`` of any depth.

- **1** - countries
- **2** - states/departments/...
- **3** - counties/municipalities/...
- **4** - subdivisions/towns/...

In [ ]:
from openplaces.api import get_admin
from openplaces.io.admin import get_admin1_iso, get_admin2_iso
from openplaces.io.ingester import Ingester
from openplaces.path import recipe_path
from openplaces.recipe import get_recipe_by_id
from openplaces.timing import get_timer
from openplaces.utils import pretty_print

In [ ]:
ADMIN_SPINE = 'admin-openplaces-2026'

ADMIN_RECIPE_IDS = {
    1: 'admin-gadm-4~1_admin1',
    2: 'admin-gadm-4~1_admin2',
    3: 'admin-gadm-4~1_admin3',
    4: 'admin-gadm-4~1_admin4',
}

In [ ]:
# Reprocess downloaded data if output files exist?
REPROCESS = True

# Redownload (and reprocess) data if downloaded files exist?
REDOWNLOAD = False

# Delete unzipped files in heap directory after processing all layers?
DELETE_UNZIPPED = True

# Save the output as the canonical list (spine) of administrative IDs?
# Rarely needed. Will overwrite country-level corrections and updates
CREATE_ADMIN_SPINE = False

# `admin1`: countries / territories
The highest level of the administrative hierarchy.
## ISO countries
Top-level administrative identifiers, gap-filled to match GADM, ships with `openplaces`

In [ ]:
get_admin1_iso().sample(5).sort_index()

## GADM level 0
GADM starts at 0 = countries (openplaces 1 = countries)

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[1]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[1], verbose=True)
ingester.ingest(reprocess=REPROCESS, redownload=REDOWNLOAD, delete_unzipped=False)

In [ ]:
# Save as initial `openplaces` Admin-1 recipe
if CREATE_ADMIN_SPINE:
    openplaces_recipe_path = recipe_path(None, ADMIN_SPINE, filename='admin1.csv')
    openplaces_recipe_path.parent.mkdir(parents=True, exist_ok=True)
    get_admin(level=1, recipe=ADMIN_RECIPE_IDS[1], all_columns=True).to_csv(
        openplaces_recipe_path,
        encoding='utf-8-sig',
    )

In [ ]:
# Read result (GADM is default geometry)
admin1 = get_admin(level=1, geom=True)
admin1.head()

# ``admin2``: states / departments

## ISO states / departments

In [ ]:
admin2_iso = get_admin2_iso()
admin2_iso.sample(5).sort_index()

## GADM states / departments

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[2]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[2], verbose=True)
ingester.ingest(reprocess=REPROCESS, delete_unzipped=False)

In [ ]:
# Save as initial `openplaces` Admin-2 recipe
if CREATE_ADMIN_SPINE:
    ADMIN2_COLUMNS = [
        'name',
        'type',
        'name_original',
        'name_alternatives',
        'type_orginal',
        'admin2_id_gadm',
        'admin2_id_original',
        'admin2_id_source',
    ]
    get_admin(level=2, recipe=ADMIN_RECIPE_IDS[2], columns=ADMIN2_COLUMNS).replace(
        'NA', ''
    ).to_csv(
        recipe_path(None, ADMIN_SPINE, filename='admin2.csv'),
        encoding='utf-8-sig',
    )

In [ ]:
admin2 = get_admin(level=2, geom=True)
admin2.head()

# ``admin3``: counties / municipalities

## GADM counties / municipalities

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[3]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[3], verbose=True)
ingester.ingest(reprocess=REPROCESS, delete_unzipped=False)

In [ ]:
# Save as initial `openplaces` Admin-3 recipe

if CREATE_ADMIN_SPINE:

    ADMIN3_COLUMNS = [
        'name',
        'type',
        'name_original',
        'name_alternatives',
        'type_orginal',
        'admin3_id_gadm',
        'admin3_id_original',
        'admin3_id_source',
    ]

    get_admin(level=3, recipe=ADMIN_RECIPE_IDS[3], columns=ADMIN3_COLUMNS).replace(
        'NA', ''
    ).to_csv(
        recipe_path(None, ADMIN_SPINE, filename='admin3.csv'),
        encoding='utf-8-sig',
    )

In [ ]:
admin3 = get_admin(level=3, all_columns=True)
admin3.head()

# ``admin4``: towns / county subdivisions / 

## GADM towns / county subdivisions / ...

In [ ]:
pretty_print(get_recipe_by_id(ADMIN_RECIPE_IDS[4]))

In [ ]:
ingester = Ingester(ADMIN_RECIPE_IDS[4], verbose=True)
ingester.ingest(reprocess=REPROCESS, delete_unzipped=DELETE_UNZIPPED)

## Testing: town-level Admin ID algorithm

In [ ]:
admin3 = get_admin(level=3, recipe=ADMIN_RECIPE_IDS[3], all_columns=True)
admin3.fillna('')

In [ ]:
import pandas as pd

from openplaces.io import read_parquet
from openplaces.path import cache_path

admin4_recipe = get_recipe_by_id(ADMIN_RECIPE_IDS[4])
admin4_path = cache_path(
    admin4_recipe['admin_id'],
    admin4_recipe['entity'],
    filename=admin4_recipe['cache_filename'],
)
admin4 = read_parquet(admin4_path)
admin4['admin3_name'] = admin4['admin3_name'].replace('NA', None)

# Test join on GADM to track dropped units
_admin4 = admin4.join(
    get_admin(level=3, all_columns=True)
    .reset_index()
    .set_index('admin3_id_gadm')['admin3_id'],
    on='admin3_id_gadm',
)

mask_dropped_and_no_lake = (
    _admin4['admin3_id'].isnull()
    & _admin4['admin3_name'].notnull()
    & ~_admin4['admin3_name'].fillna('').str.contains('Lake')
)

print(
    '\nAdmin-4 units without link to Admin-3 units (currently dropped):'
    '\n\n'
    + '\n'.join(
        _admin4[mask_dropped_and_no_lake][
            ['admin1_name', 'admin2_name', 'admin3_name', 'name']
        ].apply(' > '.join, axis=1)
    )
    + '\n'
)

admin4 = admin4.join(
    admin3.reset_index().set_index('admin3_id_gadm')['admin3_id'],
    on='admin3_id_gadm',
    how='inner',
)

`generate_admin_ids` takes a while for global towns: 2.5 minutes on a fast machine

In [ ]:
from openplaces.io.admin import generate_admin_ids

admin4['name'] = admin4['name'].replace('NA', None)
admin4 = generate_admin_ids(
    admin4,
    new_admin_id_col='admin4_id',
    parent_admin_id_col='admin3_id',
    name_long_col='name_alternatives',
)

In [ ]:
# Save as initial `openplaces` Admin-4 recipe

if CREATE_ADMIN_SPINE:

    ADMIN4_COLUMNS = [
        'name',
        'type',
        'name_original',
        'name_alternatives',
        'type_original',
        'admin4_id_original',
        'admin4_id_gadm',
        'admin4_id_source',
    ]
    admin4[ADMIN4_COLUMNS].replace('NA', '').to_csv(
        recipe_path(None, ADMIN_SPINE, filename='admin4.csv'),
        encoding='utf-8-sig',
    )